<a href="https://colab.research.google.com/github/nithzsenpai/mL-lab/blob/main/1BM23CS219_LAB10_PCAipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/heart.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [2]:
# Display basic information about the DataFrame
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


None

From the `df.info()` output, we can observe that there are no missing values. Next, we will identify numerical and categorical features for preprocessing. We need to decide which columns are truly categorical and which are numerical.

Based on common heart disease datasets and column names:
*   `Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope` are likely categorical or ordinal features.
*   `Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`, `Oldpeak` are likely numerical features.

The target variable is `HeartDisease`.

In [8]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define categorical and numerical columns based on common understanding of the dataset
categorical_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
numerical_cols = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']

# The target variable is 'HeartDisease'
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# Apply One-Hot Encoding to categorical columns
# We will use ColumnTransformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols), # Scale numerical features
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols) # One-hot encode categorical features
    ],
    remainder='passthrough'
)

# Fit and transform the data
X_processed = preprocessor.fit_transform(X)

# Get feature names after one-hot encoding for better interpretability (optional)
onehot_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(onehot_feature_names)

X_processed_df = pd.DataFrame(X_processed, columns=all_feature_names)

display(X_processed_df.head())

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_F,Sex_M,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,-1.433140,0.410909,0.825070,-0.551341,1.382928,-0.832432,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1,-0.478484,1.491752,-0.171961,-0.551341,0.754157,0.105664,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,-1.751359,-0.129513,0.770188,-0.551341,-1.525138,-0.832432,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
3,-0.584556,0.302825,0.139040,-0.551341,-1.132156,0.574711,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0.051881,0.951331,-0.034755,-0.551341,-0.581981,-0.832432,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


Now that our data is preprocessed (scaled numerical features and one-hot encoded categorical features), we can proceed to build and evaluate classification models. We will start by splitting the data into training and testing sets.

In [9]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (734, 20)
Testing data shape: (184, 20)


Next, let's build and evaluate classification models using SVM, Logistic Regression, and Random Forest. We will compare their accuracy on the test set.

In [10]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Initialize models
svm_model = SVC(random_state=42)
logreg_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' for small datasets
rf_model = RandomForestClassifier(random_state=42)

models = {
    'SVM': svm_model,
    'Logistic Regression': logreg_model,
    'Random Forest': rf_model
}

model_accuracies = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    model_accuracies[name] = accuracy
    print(f"{name} Accuracy: {accuracy:.4f}")

# Display all accuracies
print("\n--- Model Accuracies (without PCA) ---")
for name, accuracy in model_accuracies.items():
    print(f"{name}: {accuracy:.4f}")


Training SVM...
SVM Accuracy: 0.8696

Training Logistic Regression...
Logistic Regression Accuracy: 0.8533

Training Random Forest...
Random Forest Accuracy: 0.8804

--- Model Accuracies (without PCA) ---
SVM: 0.8696
Logistic Regression: 0.8533
Random Forest: 0.8804


Now, let's apply Principal Component Analysis (PCA) to reduce the dimensionality of our preprocessed data and then retrain the models to observe the impact on accuracy. We will choose a number of components that captures a significant portion of the variance (e.g., 95%).

In [11]:
from sklearn.decomposition import PCA

# Apply PCA
# We will choose n_components to explain 95% of the variance
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Original number of features: {X_train.shape[1]}")
print(f"Reduced number of features after PCA (explaining 95% variance): {X_train_pca.shape[1]}")

# Retrain models with PCA-transformed data
model_accuracies_pca = {}

for name, model in models.items(): # Use the same initialized models
    print(f"\nRetraining {name} with PCA data...")
    model.fit(X_train_pca, y_train)
    y_pred_pca = model.predict(X_test_pca)
    accuracy_pca = accuracy_score(y_test, y_pred_pca)
    model_accuracies_pca[name] = accuracy_pca
    print(f"{name} Accuracy (with PCA): {accuracy_pca:.4f}")

# Display and compare accuracies
print("\n--- Comparison of Model Accuracies ---")
print("Model                   | Without PCA | With PCA")
print("------------------------|-------------|----------")
for name in models.keys():
    print(f"{name:<23} | {model_accuracies[name]:<11.4f} | {model_accuracies_pca[name]:<8.4f}")

Original number of features: 20
Reduced number of features after PCA (explaining 95% variance): 12

Retraining SVM with PCA data...
SVM Accuracy (with PCA): 0.8587

Retraining Logistic Regression with PCA data...
Logistic Regression Accuracy (with PCA): 0.8533

Retraining Random Forest with PCA data...
Random Forest Accuracy (with PCA): 0.8641

--- Comparison of Model Accuracies ---
Model                   | Without PCA | With PCA
------------------------|-------------|----------
SVM                     | 0.8696      | 0.8587  
Logistic Regression     | 0.8533      | 0.8533  
Random Forest           | 0.8804      | 0.8641  


This comparison shows the impact of PCA on the accuracy of each model. As you can see, sometimes accuracy might decrease slightly with PCA, but the benefit lies in reduced computational complexity due to fewer features, which is a trade-off often considered in real-world scenarios.